<a href="https://colab.research.google.com/github/Sripramod-droid/Skill-Development-Course/blob/main/sample_analaysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
Fine-tuning GPT-2 on a Custom Movie Script Dataset in Google Colab.
"""

# @title ## 1. Setup Environment
# Install necessary libraries
!pip install transformers[torch] datasets accelerate -q

import os
import torch
import re
from datasets import load_dataset, Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    TextDataset,
    DataCollatorForLanguageModeling,
    pipeline
)
from google.colab import files

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU. Fine-tuning will be very slow.")
    print("Consider switching Runtime type (Runtime -> Change runtime type -> GPU)")

# @title ## 2. Upload Your Movie Script Dataset
# Ensure you upload a single .txt file containing your movie script(s)
print("Please upload your movie script dataset (.txt file):")
uploaded = files.upload()

# Get the filename of the uploaded file
if not uploaded:
    raise ValueError("No file uploaded. Please run the cell again and upload your .txt file.")
else:
    # Assuming only one file is uploaded
    input_file_path = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded '{input_file_path}'")

    # Optional: Display the first few lines of the file
    print("\nFirst 5 lines of the uploaded file:")
    with open(input_file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(line.strip())

# @title ## 3. Configuration
# --- Model Configuration ---
base_model_name = "gpt2" # You can try "gpt2-medium" for a larger model if resources allow

# --- Data Configuration ---
# input_file_path is already set from the upload step
output_dir = "./gpt2-finetuned-moviescript" # Directory to save the fine-tuned model
model_save_path = os.path.join(output_dir, "final_model")

# --- Training Hyperparameters ---
num_train_epochs = 3 # @param {type:"integer"}
per_device_train_batch_size = 4 # @param {type:"integer"} # Reduce if you encounter CUDA out-of-memory errors
per_device_eval_batch_size = 4 # @param {type:"integer"}
learning_rate = 5e-5 # @param {type:"number"}
warmup_steps = 100 # @param {type:"integer"}
weight_decay = 0.01 # @param {type:"number"}
logging_steps = 50 # @param {type:"integer"} # Log training loss every N steps
save_steps = 500 # @param {type:"integer"} # Save checkpoint every N steps (optional, adjust based on dataset size)
block_size = 128  # @param {type:"integer"} # Sequence length for model input. Max is 1024 for GPT-2. Reduce if memory issues occur.

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(model_save_path, exist_ok=True)

print(f"\n--- Configuration Summary ---")
print(f"Base Model: {base_model_name}")
print(f"Input Dataset: {input_file_path}")
print(f"Output Directory: {output_dir}")
print(f"Model Save Path: {model_save_path}")
print(f"Epochs: {num_train_epochs}")
print(f"Batch Size: {per_device_train_batch_size}")
print(f"Learning Rate: {learning_rate}")
print(f"Block Size (Sequence Length): {block_size}")
print("-" * 30)

# @title ## 4. Load Tokenizer and Base Model
print(f"\nLoading tokenizer and model for '{base_model_name}'...")

tokenizer = GPT2Tokenizer.from_pretrained(base_model_name)
model = GPT2LMHeadModel.from_pretrained(base_model_name)

# --- Set Padding Token ---
# GPT-2 doesn't have a default pad token, set it to eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id
    print("Set pad_token to eos_token")

print("Tokenizer and model loaded successfully.")


# @title ## 5. Load and Prepare Dataset
print("\nLoading and preparing dataset...")

# Load dataset using Hugging Face's datasets library
# This is generally more flexible and memory-efficient than TextDataset for large files
raw_datasets = load_dataset('text', data_files={'train': input_file_path})
print(f"\nRaw dataset loaded: {raw_datasets}")

# --- Tokenization Function ---
def tokenize_function(examples):
    # Tokenize the text. The tokenizer handles adding special tokens if configured.
    return tokenizer(examples["text"], truncation=False) # Don't truncate yet, group_texts will handle length

print("\nTokenizing dataset...")
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4, # Use multiple processes for faster tokenization
    remove_columns=["text"] # Remove the original text column
)
print(f"\nTokenized dataset sample (first sequence): {tokenized_datasets['train'][0]['input_ids'][:50]}...")

# --- Grouping Function (Main data processing step) ---
# Concatenate all texts and then split into chunks of block_size
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # Drop the small remainder to ensure all chunks are of block_size
    total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    # Create the 'labels' column (GPT-2 uses input_ids as labels for LM)
    result["labels"] = result["input_ids"].copy()
    return result

print(f"\nGrouping texts into blocks of size {block_size}...")
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=4, # Use multiple processes
)

print(f"\nProcessed dataset ready for training.")
print(f"Number of sequences in training set: {len(lm_datasets['train'])}")
print(f"Example sequence length: {len(lm_datasets['train'][0]['input_ids'])}")


# @title ## 6. Fine-Tuning the Model

print("\n--- Starting Fine-Tuning ---")

# --- Define Training Arguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    # per_device_eval_batch_size=per_device_eval_batch_size, # Evaluation not strictly needed for generation focus
    learning_rate=learning_rate,
    warmup_steps=warmup_steps,
    weight_decay=weight_decay,
    logging_dir='./logs',
    logging_steps=logging_steps,
    save_steps=save_steps,
    save_total_limit=2, # Only keep the last 2 checkpoints
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU is available
    report_to="none", # Disable reporting to wandb/tensorboard for simplicity
    # evaluation_strategy="steps", # Enable if you have an eval dataset and want metrics
    # eval_steps=save_steps,       # Evaluate every save_steps
)

# --- Initialize Trainer ---
# Data Collator for Language Modeling automatically handles padding and labels
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # This is for Causal LM (like GPT-2), not Masked LM (like BERT)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    # eval_dataset=lm_datasets["validation"], # Provide validation set if you have one
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# --- Start Training ---
print("\nTraining in progress... This may take a while.")
try:
    train_result = trainer.train()

    # --- Save Training Metrics ---
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)

    print("\n--- Training Completed ---")

    # --- Save the Fine-tuned Model and Tokenizer ---
    print(f"\nSaving fine-tuned model and tokenizer to {model_save_path}...")
    trainer.save_model(model_save_path) # Saves the model weights and config
    tokenizer.save_pretrained(model_save_path) # Saves the tokenizer
    print("Model and tokenizer saved successfully.")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")
    print("Please check hyperparameters (e.g., reduce batch size or block size if it's a memory issue).")


# @title ## 7. Test Generation and Compare Models

print("\n--- Testing Text Generation ---")

# --- Load the Fine-tuned Model ---
print(f"\nLoading the fine-tuned model from {model_save_path}...")
try:
    ft_model = GPT2LMHeadModel.from_pretrained(model_save_path).to(device)
    ft_tokenizer = GPT2Tokenizer.from_pretrained(model_save_path)
    print("Fine-tuned model loaded.")
except Exception as e:
    print(f"Error loading fine-tuned model: {e}")
    print("Skipping fine-tuned model generation.")
    ft_model = None
    ft_tokenizer = None

# --- Load the Original Base Model ---
print(f"\nLoading the original base model '{base_model_name}' for comparison...")
try:
    base_model = GPT2LMHeadModel.from_pretrained(base_model_name).to(device)
    base_tokenizer = GPT2Tokenizer.from_pretrained(base_model_name)
    print("Original base model loaded.")
except Exception as e:
    print(f"Error loading base model: {e}")
    print("Skipping base model generation.")
    base_model = None
    base_tokenizer = None


# --- Generation Helper Function ---
def generate_text(model, tokenizer, prompt, max_length=150, num_sequences=1):
    if model is None or tokenizer is None:
        return ["Model not available."]

    print(f"\nGenerating text with prompt: '{prompt}'")
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Set pad token id for generation if needed (though for single prompt it might not be)
    if tokenizer.pad_token_id is None:
      effective_pad_token_id = tokenizer.eos_token_id
    else:
      effective_pad_token_id = tokenizer.pad_token_id

    # Generation parameters
    outputs = model.generate(
        inputs,
        max_length=max_length,
        num_return_sequences=num_sequences,
        do_sample=True,        # Enable sampling for more creative output
        temperature=0.8,       # Controls randomness (lower = more focused)
        top_k=50,              # Considers only the top K tokens
        top_p=0.95,            # Nucleus sampling (considers cumulative probability)
        pad_token_id=effective_pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2 # Prevent repeating short phrases
    )

    generated_texts = []
    for i, output in enumerate(outputs):
        text = tokenizer.decode(output, skip_special_tokens=True)
        # Simple post-processing: remove the prompt from the beginning if present
        if text.startswith(prompt):
           text = text[len(prompt):].strip()
        # Clean up potential artifacts or incomplete sentences at the end
        text = re.sub(r'\n\n.*', '', text) # Remove multiple newlines often indicating end
        text = text.rsplit('.', 1)[0] + '.' if '.' in text else text # Try to end on a sentence
        generated_texts.append(f"--- Sequence {i+1} ---\n{prompt}{text}\n") # Add prompt back for context

    return generated_texts


# --- Define Prompt and Generate ---
# <<< CHANGE THIS PROMPT to something relevant to your movie script data! >>>
# Good prompts might be Scene Headings, Character names followed by (CONT'D), or initial dialogue.
test_prompt = "INT. COFFEE SHOP - DAY"
# test_prompt = "DETECTIVE MILLER (V.O.)"
# test_prompt = "Rain lashes against the window."

print("\n" + "="*50)
print("GENERATING WITH FINE-TUNED MODEL")
print("="*50)
if ft_model and ft_tokenizer:
  ft_output = generate_text(ft_model, ft_tokenizer, test_prompt, max_length=200, num_sequences=2)
  for out in ft_output:
      print(out)
else:
  print("Fine-tuned model generation skipped due to loading error.")


print("\n" + "="*50)
print("GENERATING WITH ORIGINAL BASE MODEL")
print("="*50)
if base_model and base_tokenizer:
  base_output = generate_text(base_model, base_tokenizer, test_prompt, max_length=200, num_sequences=2)
  for out in base_output:
      print(out)
else:
  print("Base model generation skipped due to loading error.")


print("\n--- Comparison Complete ---")
print("Compare the outputs above. The fine-tuned model should ideally produce text")
print("that is more consistent with the style, format, characters, and vocabulary")
print("found in your movie script dataset.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Saving sample_script.txt to sample_script.txt

Successfully uploaded 'sample_script.txt'

First 5 lines of the uploaded file:
FADE IN:

INT. DUSTY BOOKSTORE - DAY

Sunlight streams through a grimy window, illuminating floating dust motes. Books are piled high on shelves, tables, even the floor. The air smells of old paper and neglect.

--- Configuration Summary ---
Base Model: gpt2
Input Dataset: sample_script.txt
Output Directory: ./gpt2-finetuned-moviescript
Model Save Path: ./gpt2-finetuned-moviescript/final_model
Epochs: 3
Batch Size: 4
Learning Rate: 5e-05
Block Size (Sequence Length): 128
------------------------------

Loading tokenizer and model for 'gpt2'...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

<ipython-input-1-07edbbb94e5b>:188: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


***** train metrics *****
  epoch                    =        3.0
  total_flos               =      730GF
  train_loss               =     4.3875
  train_runtime            = 0:00:52.21
  train_samples_per_second =       0.23
  train_steps_per_second   =      0.057

--- Training Completed ---

Saving fine-tuned model and tokenizer to ./gpt2-finetuned-moviescript/final_model...
Model and tokenizer saved successfully.

--- Testing Text Generation ---

Loading the fine-tuned model from ./gpt2-finetuned-moviescript/final_model...
Fine-tuned model loaded.

Loading the original base model 'gpt2' for comparison...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Original base model loaded.

GENERATING WITH FINE-TUNED MODEL

Generating text with prompt: 'INT. COFFEE SHOP - DAY'
--- Sequence 1 ---
INT. COFFEE SHOP - DAY.
T.O.S.H. - T.C. RICHARDSON. HOMESHOE. NICHOLAS. STANLEY. BARBER. CHECKER. ARTHUR. BURKE. BOOMBUCKER .
- CHAIRMAN. HEY. THE HURT MATCH. I know the word. BUDDY'S BEAR. SHE'LL BEAT HER. E.J. DUTCHER' SIDE. SICK. SHADY GORDON'N. JONES. JACK. KASTA. LEN. O'NEILL. MARY. MONTANA. DONALD. ZEALAND. TONY. THOMAS (TEN DAYS ago, as the old adage goes). TOLD you that.

--- Sequence 2 ---
INT. COFFEE SHOP - DAYWe've got a lot of cool stuff out there! Check out the best in home brewing.
. . .
, __________________
-
The Daily Caller
's Ben Cargill's Top 10 Recipes - The '60s
This is the first post in a series of posts on 'top 10' recipes. The list below is by no means exhaustive, and may not be complete. See all of the posts here . (You can read the full list of 'Top 10 recipes' below.) .


GENERATING WITH ORIGINAL BASE MODEL

Generating text w